In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightkurve as lk
import time

c:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [3]:
PROJECT_ROOT = Path.cwd().parent

CSV_PATH = PROJECT_ROOT / "cumulative_2026.08.12_23.25.19.csv"
FITS_DIR = PROJECT_ROOT / "data" / "raw" / "fits"

FITS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("CSV path:", CSV_PATH)
print("FITS directory:", FITS_DIR)

Project root: d:\exoplanet-astro
CSV path: d:\exoplanet-astro\cumulative_2026.08.12_23.25.19.csv
FITS directory: d:\exoplanet-astro\data\raw\fits


In [4]:
print("CSV exists:", CSV_PATH.exists())

CSV exists: True


In [5]:
with open(CSV_PATH, "r", encoding="utf-8") as f:
    for i in range(20):
        print(i, repr(f.readline().strip()))
        

0 '# This file was produced by the NASA Exoplanet Archive  http://exoplanetarchive.ipac.caltech.edu'
1 '# Wed Aug 12 23:25:19 2026'
2 '#'
3 '# COLUMN kepid:          KepID'
4 '# COLUMN kepoi_name:     KOI Name'
5 '# COLUMN kepler_name:    Kepler Name'
6 '# COLUMN koi_disposition: Exoplanet Archive Disposition'
7 '# COLUMN koi_vet_stat:   Vetting Status'
8 '# COLUMN koi_vet_date:   Date of Last Parameter Update'
9 '# COLUMN koi_pdisposition: Disposition Using Kepler Data'
10 '# COLUMN koi_score:      Disposition Score'
11 '# COLUMN koi_fpflag_nt:  Not Transit-Like False Positive Flag'
12 '# COLUMN koi_fpflag_ss:  Stellar Eclipse False Positive Flag'
13 '# COLUMN koi_fpflag_co:  Centroid Offset False Positive Flag'
14 '# COLUMN koi_fpflag_ec:  Ephemeris Match Indicates Contamination False Positive Flag'
15 '# COLUMN koi_disp_prov:  Disposition Provenance'
16 '# COLUMN koi_comment:    Comment'
17 '# COLUMN koi_period:     Orbital Period [days]'
18 '# COLUMN koi_period_err1: Orbital Period

In [6]:
with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    for i, line in enumerate(f):
        if "kepid" in line and "kepoi_name" in line and "koi_disposition" in line:
            print("Header found at line:", i)
            print(line.strip())
            break

Header found at line: 144
rowid,kepid,kepoi_name,kepler_name,koi_disposition,koi_vet_stat,koi_vet_date,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_disp_prov,koi_comment,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_time0,koi_time0_err1,koi_time0_err2,koi_eccen,koi_eccen_err1,koi_eccen_err2,koi_longp,koi_longp_err1,koi_longp_err2,koi_impact,koi_impact_err1,koi_impact_err2,koi_duration,koi_duration_err1,koi_duration_err2,koi_ingress,koi_ingress_err1,koi_ingress_err2,koi_depth,koi_depth_err1,koi_depth_err2,koi_ror,koi_ror_err1,koi_ror_err2,koi_srho,koi_srho_err1,koi_srho_err2,koi_fittype,koi_prad,koi_prad_err1,koi_prad_err2,koi_sma,koi_sma_err1,koi_sma_err2,koi_incl,koi_incl_err1,koi_incl_err2,koi_teq,koi_teq_err1,koi_teq_err2,koi_insol,koi_insol_err1,koi_insol_err2,koi_dor,koi_dor_err1,koi_dor_err2,koi_limbdark_mod,koi_ldm_coeff4,koi_ldm_coeff3,koi_ldm_coeff2,koi_ldm_coeff1,koi_parm_prov,koi_max_sng

In [7]:
df = pd.read_csv(CSV_PATH, skiprows=144)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Shape: (9564, 141)

First 5 rows:


,rowid,kepid,kepoi_name,kepler_name,koi_disposition,koi_vet_stat,koi_vet_date,koi_pdisposition,koi_score,koi_fpflag_nt,...,koi_dicco_mdec,koi_dicco_mdec_err,koi_dicco_msky,koi_dicco_msky_err,koi_dikco_mra,koi_dikco_mra_err,koi_dikco_mdec,koi_dikco_mdec_err,koi_dikco_msky,koi_dikco_msky_err
0,1,10797460,K00752.01,Kepler-227 b,CONFIRMED,Done,2018-08-16,CANDIDATE,1.000,0,...,0.200,0.160,0.200,0.170,0.080,0.130,0.310,0.170,0.320,0.160
1,2,10797460,K00752.02,Kepler-227 c,CONFIRMED,Done,2018-08-16,CANDIDATE,0.969,0,...,0.000,0.480,0.390,0.360,0.490,0.340,0.120,0.730,0.500,0.450
2,3,10811496,K00753.01,NaN,CANDIDATE,Done,2018-08-16,CANDIDATE,0.000,0,...,-0.034,0.070,0.042,0.072,0.002,0.071,-0.027,0.074,0.027,0.074
3,4,10848459,K00754.01,NaN,FALSE POSITIVE,Done,2018-08-16,FALSE POSITIVE,0.000,0,...,0.147,0.078,0.289,0.079,-0.257,0.072,0.099,0.077,0.276,0.076
4,5,10854555,K00755.01,Kepler-664 b,CONFIRMED,Done,2018-08-16,CANDIDATE,1.000,0,...,-0.090,0.180,0.100,0.140,0.070,0.180,0.020,0.160,0.070,0.200


In [8]:
print("KIC column exists:", "kepid" in df.columns)
print("Disposition column exists:", "koi_disposition" in df.columns)

print("\nUnique KIC IDs:", df["kepid"].nunique())
print("Missing KIC IDs:", df["kepid"].isna().sum())

print("\nDisposition counts:")
print(df["koi_disposition"].value_counts())

KIC column exists: True
Disposition column exists: True

Unique KIC IDs: 8214
Missing KIC IDs: 0

Disposition counts:
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2747
CANDIDATE         1978
Name: count, dtype: int64


In [9]:
kic_ids = (
    df["kepid"]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)

print("Total unique KIC IDs:", len(kic_ids))
print("First 10 IDs:", kic_ids[:10])

Total unique KIC IDs: 8214
First 10 IDs: [10797460, 10811496, 10848459, 10854555, 10872983, 6721123, 10910878, 11446443, 10666592, 6922244]


In [10]:
kic_ids = sorted(kic_ids)

mid = len(kic_ids) // 2

jhalak_ids = kic_ids[:mid]
gayatri_ids = kic_ids[mid:]

print("Total IDs:", len(kic_ids))
print("Jhalak's IDs:", len(jhalak_ids))
print("Gayatri's IDs:", len(gayatri_ids))

print("\nJhalak's ID range:", jhalak_ids[0], "to", jhalak_ids[-1])
print("Gayatri's ID range:", gayatri_ids[0], "to", gayatri_ids[-1])

Total IDs: 8214
Jhalak's IDs: 4107
Gayatri's IDs: 4107

Jhalak's ID range: 757450 to 7887791
Gayatri's ID range: 7887890 to 12935144


In [11]:
METADATA_DIR = PROJECT_ROOT / "data" / "raw" / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

jhalak_ids_path = METADATA_DIR / "jhalak_kic_ids.csv"
gayatri_ids_path = METADATA_DIR / "gayatri_kic_ids.csv"

pd.DataFrame({"kepid": jhalak_ids}).to_csv(
    jhalak_ids_path,
    index=False
)

pd.DataFrame({"kepid": gayatri_ids}).to_csv(
    gayatri_ids_path,
    index=False
)

print("Jhalak's IDs saved to:", jhalak_ids_path)
print("Gayatri's IDs saved to:", gayatri_ids_path)

Jhalak's IDs saved to: d:\exoplanet-astro\data\raw\metadata\jhalak_kic_ids.csv
Gayatri's IDs saved to: d:\exoplanet-astro\data\raw\metadata\gayatri_kic_ids.csv


In [12]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

FITS_DIR = PROJECT_ROOT / "data" / "raw" / "fits"
TEMP_DIR = PROJECT_ROOT / "data" / "raw" / "mast_temp"
METADATA_DIR = PROJECT_ROOT / "data" / "raw" / "metadata"

print("Project root:", PROJECT_ROOT)

print("\nFITS directory:")
print(FITS_DIR)
print("Exists:", FITS_DIR.exists())

if FITS_DIR.exists():
    fits_files = list(FITS_DIR.rglob("*.fits"))
    print("FITS files found:", len(fits_files))

    for f in fits_files[:20]:
        print(f)

print("\nTemporary directory:")
print(TEMP_DIR)
print("Exists:", TEMP_DIR.exists())

if TEMP_DIR.exists():
    temp_files = [
        f for f in TEMP_DIR.rglob("*")
        if f.is_file()
    ]

    print("Temporary files found:", len(temp_files))

    for f in temp_files[:20]:
        print(f)

Project root: d:\exoplanet-astro

FITS directory:
d:\exoplanet-astro\data\raw\fits
Exists: True
FITS files found: 41503
d:\exoplanet-astro\data\raw\fits\kplr000757450-2009166043257_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2009259160929_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2009350155506_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2010078095331_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2010174085026_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2010265121752_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2010355172524_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2011073133259_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2011177032512_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2011208035123_slc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2011240104155_slc.fits
d:\exoplanet-astro\data\raw\fits\kplr000757450-2011271113734_llc.fits
d:\exoplanet-astro\data\raw\fits\kplr000

In [13]:
from pathlib import Path
from astropy.io import fits

temp_fits = list(TEMP_DIR.rglob("*.fits"))

print("Total FITS files in TEMP_DIR:", len(temp_fits))

# Count by parent/source folder
source_counts = {}

for file in temp_fits:
    parts = file.parts

    if "mastDownload" in parts:
        idx = parts.index("mastDownload")

        if idx + 1 < len(parts):
            source = parts[idx + 1]
        else:
            source = "unknown"
    else:
        source = "unknown"

    source_counts[source] = source_counts.get(source, 0) + 1

print("\nFiles by source:")
for source, count in source_counts.items():
    print(f"{source}: {count}")

# Check file sizes
total_bytes = sum(file.stat().st_size for file in temp_fits)

print(
    f"\nTotal size: "
    f"{total_bytes / (1024**3):.2f} GB"
)

print(
    f"Average size: "
    f"{total_bytes / len(temp_fits) / (1024**2):.2f} MB"
)

# Show first 20 files
print("\nFirst 20 FITS files:")
for file in temp_fits[:20]:
    print(file)

Total FITS files in TEMP_DIR: 522

Files by source:
Kepler: 1
HLSP: 521

Total size: 0.09 GB
Average size: 0.17 MB

First 20 FITS files:
d:\exoplanet-astro\data\raw\mast_temp\mastDownload\Kepler\kplr006032920_sc_Q000000000000003000\kplr006032920-2012277125453_slc.fits
d:\exoplanet-astro\data\raw\mast_temp\mastDownload\HLSP\hlsp_iris_kepler_kepler_kplr002436378-q01_kepler_v1.0_lc\hlsp_iris_kepler_kepler_kplr002436378-q01_kepler_v1.0_lc.fits
d:\exoplanet-astro\data\raw\mast_temp\mastDownload\HLSP\hlsp_iris_kepler_kepler_kplr002436378-q02_kepler_v1.0_lc\hlsp_iris_kepler_kepler_kplr002436378-q02_kepler_v1.0_lc.fits
d:\exoplanet-astro\data\raw\mast_temp\mastDownload\HLSP\hlsp_iris_kepler_kepler_kplr002436378-q03_kepler_v1.0_lc\hlsp_iris_kepler_kepler_kplr002436378-q03_kepler_v1.0_lc.fits
d:\exoplanet-astro\data\raw\mast_temp\mastDownload\HLSP\hlsp_iris_kepler_kepler_kplr002436378-q04_kepler_v1.0_lc\hlsp_iris_kepler_kepler_kplr002436378-q04_kepler_v1.0_lc.fits
d:\exoplanet-astro\data\raw\mas

In [14]:
# Inspect what Lightkurve returns for one KIC

test_kic = 1870398

search_result = lk.search_lightcurve(
    f"KIC {test_kic}",
    mission="Kepler"
)

print("Total products returned:", len(search_result))

display(search_result.table)

Total products returned: 17


intentType,obs_collection,provenance_name,instrument_name,project,filters,wave_region,target_name,target_classification,obs_id,s_ra,s_dec,dataproduct_type,proposal_pi,calib_level,t_min,t_max,t_exptime,wavelength_region,em_min,em_max,obs_title,t_obs_release,proposal_id,proposal_type,sequence_number,s_region,jpegURL,dataURL,dataRights,mtFlag,srcDen,obsid,objID,wave_min,wave_max,exptime,distance,obsID,obs_collection_products,dataproduct_type_products,description,type,dataURI,productType,productGroupDescription,productSubGroupDescription,productDocumentationURL,project_products,prvversion,proposal_id_products,productFilename,size,parent_obsid,dataRights_products,calib_level_products,filters_products,author,mission,#,year,sort_order
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,s,arcsec,,,,,,,,,,,,,,,,,,,,,,,,
str7,str6,str10,str6,str6,str6,str7,str13,str1,str68,float64,float64,str10,str15,int64,float64,float64,float64,str7,float64,float64,str18,float64,str2,str6,int64,str112,str115,str121,str6,bool,float64,str8,str9,float64,float64,float64,float64,str8,str6,str10,str59,str1,str125,str9,str28,str8,str1,str10,str3,str2,str77,int64,str8,str6,int64,str6,str10,object,object,int64,int64
science,Kepler,Kepler,Kepler,Kepler,KEPLER,OPTICAL,kplr001870398,--,kplr001870398_lc_Q011111111111111111,291.91556,37.32258,timeseries,William Borucki,2,54964.01041666667,56423.50069444445,1800.0,OPTICAL,418.4,905.0,Kepler Key Project,nan,EX,Kepler,--,POLYGON ICRS 291.91276577 37.32480200 291.91276594 37.32035800 291.91835406 37.32035800 291.91835423 37.32480200,http://archive.stsci.edu/missions/kepler/previews/0018/001870398/kplr001870398-2013098041711_llc_bw_large.png,http://archive.stsci.edu/missions/kepler/lightcurves/0018/001870398/kplr001870398_lc_Q011111111111111111.tar,PUBLIC,--,5885.0,537747,764625,418.4,905.0,1800,0.0,537747,Kepler,timeseries,Lightcurve Long Cadence (CLC) - Q1,C,mast:KEPLER/url/missions/kepler/lightcurves/0018/001870398/kplr001870398-2009166043257_llc.fits,SCIENCE,Minimum Recommended Products,LLC,--,Kepler,--,EX,kplr001870398-2009166043257_llc.fits,192960,537747,PUBLIC,2,KEPLER,Kepler,Kepler Quarter 01,None,2009,1
science,Kepler,Kepler,Kepler,Kepler,KEPLER,OPTICAL,kplr001870398,--,kplr001870398_lc_Q011111111111111111,291.91556,37.32258,timeseries,William Borucki,2,54964.01041666667,56423.50069444445,1800.0,OPTICAL,418.4,905.0,Kepler Key Project,nan,EX,Kepler,--,POLYGON ICRS 291.91276577 37.32480200 291.91276594 37.32035800 291.91835406 37.32035800 291.91835423 37.32480200,http://archive.stsci.edu/missions/kepler/previews/0018/001870398/kplr001870398-2013098041711_llc_bw_large.png,http://archive.stsci.edu/missions/kepler/lightcurves/0018/001870398/kplr001870398_lc_Q011111111111111111.tar,PUBLIC,--,5885.0,537747,764625,418.4,905.0,1800,0.0,537747,Kepler,timeseries,Lightcurve Long Cadence (CLC) - Q2,C,mast:KEPLER/url/missions/kepler/lightcurves/0018/001870398/kplr001870398-2009259160929_llc.fits,SCIENCE,Minimum Recommended Products,LLC,--,Kepler,--,EX,kplr001870398-2009259160929_llc.fits,466560,537747,PUBLIC,2,KEPLER,Kepler,Kepler Quarter 02,None,2009,1
science,Kepler,Kepler,Kepler,Kepler,KEPLER,OPTICAL,kplr001870398,--,kplr001870398_lc_Q011111111111111111,291.91556,37.32258,timeseries,William Borucki,2,54964.01041666667,56423.50069444445,1800.0,OPTICAL,418.4,905.0,Kepler Key Project,nan,EX,Kepler,--,POLYGON ICRS 291.91276577 37.32480200 291.91276594 37.32035800 291.91835406 37.32035800 291.91835423 37.32480200,http://archive.stsci.edu/missions/kepler/previews/0018/001870398/kplr001870398-2013098041711_llc_bw_large.png,http://archive.stsci.edu/missions/kepler/lightcurves/0018/001870398/kplr001870398_lc_Q011111111111111111.tar,PUBLIC,--,5885.0,537747,764625,418.4,905.0,1800,0.0,537747,Kepler,timeseries,Lightcurve Long Cadence (CLC) - Q3,C,mast:KEPLER/url/missions/kepler/lightcurves/0018/001870398/kplr001870398-2009350155506_llc.fits,SCIENCE,Minimum Recommended Products,LLC,--,Kepler,--,EX,kplr001870398-2009350155506_llc.fits,466560,537747,PU

In [15]:
from collections import Counter

print(Counter(search_result.table["author"].tolist()))

Counter({'Kepler': 17})


In [16]:
def download_kic_products(kic, max_retries=3):
    """
    Download all available RAW Kepler science light-curve FITS products
    for one KIC.

    Keeps both Long Cadence and Short Cadence products.
    Does NOT download HLSP products.
    Does NOT preprocess or modify FITS files.

    Existing final FITS files are skipped.
    Failed downloads are retried.
    """

    kic = int(kic)

    try:
        # ---------------------------------------------------------
        # 1. Search MAST for this KIC
        # ---------------------------------------------------------

        search_result = lk.search_lightcurve(
            f"KIC {kic}",
            mission="Kepler"
        )

        # ---------------------------------------------------------
        # 2. Keep ONLY actual Kepler products
        # ---------------------------------------------------------

        authors = search_result.table["author"]

        kepler_mask = [
            str(author) == "Kepler"
            for author in authors
        ]

        kepler_products = search_result[kepler_mask]

        products_found = len(kepler_products)

        if products_found == 0:
            return {
                "kepid": kic,
                "status": "no_data",
                "products_found": 0,
                "products_downloaded": 0,
                "failed_products": 0
            }

        products_downloaded = 0
        failed_products = 0

        # ---------------------------------------------------------
        # 3. Download every Kepler product
        # ---------------------------------------------------------

        for i in range(products_found):

            data_uri = str(
                kepler_products.table["dataURI"][i]
            )

            filename = data_uri.split("/")[-1]

            final_path = FITS_DIR / filename

            # -----------------------------------------------------
            # Already downloaded?
            # -----------------------------------------------------

            if (
                final_path.exists()
                and final_path.stat().st_size > 0
            ):
                print(f"Already exists: {filename}")

                products_downloaded += 1
                continue

            success = False

            # -----------------------------------------------------
            # Retry download
            # -----------------------------------------------------

            for attempt in range(1, max_retries + 1):

                try:

                    print(
                        f"KIC {kic} | "
                        f"Product {i + 1}/{products_found} | "
                        f"Attempt {attempt}/{max_retries}"
                    )

                    # -------------------------------------------------
                    # Remove ONLY an incomplete copy of THIS filename
                    # from TEMP_DIR before retrying.
                    # -------------------------------------------------

                    old_temp_files = list(
                        TEMP_DIR.rglob(filename)
                    )

                    for old_file in old_temp_files:

                        try:
                            old_file.unlink()

                        except Exception:
                            pass

                    # -------------------------------------------------
                    # Download product
                    # -------------------------------------------------

                    kepler_products[i].download(
                        download_dir=str(TEMP_DIR)
                    )

                    # -------------------------------------------------
                    # Find THIS exact downloaded FITS
                    # -------------------------------------------------

                    downloaded_files = list(
                        TEMP_DIR.rglob(filename)
                    )

                    if not downloaded_files:
                        raise FileNotFoundError(
                            f"Downloaded FITS not found: {filename}"
                        )

                    source_path = downloaded_files[0]

                    # -------------------------------------------------
                    # Make sure the file isn't empty
                    # -------------------------------------------------

                    if source_path.stat().st_size == 0:
                        raise IOError(
                            f"Downloaded FITS is empty: {filename}"
                        )

                    # -------------------------------------------------
                    # Move to FINAL raw directory
                    # -------------------------------------------------

                    shutil.move(
                        str(source_path),
                        str(final_path)
                    )

                    # -------------------------------------------------
                    # Verify final file
                    # -------------------------------------------------

                    if not final_path.exists():
                        raise IOError(
                            f"FITS was not found after moving: {filename}"
                        )

                    products_downloaded += 1
                    success = True

                    print(
                        f"Downloaded: {filename}"
                    )

                    break

                except Exception as e:

                    print(
                        f"FAILED | KIC {kic} | "
                        f"Product {i + 1}/{products_found} | "
                        f"Attempt {attempt}: {e}"
                    )

                    # Remove only this failed temporary file
                    failed_temp_files = list(
                        TEMP_DIR.rglob(filename)
                    )

                    for failed_file in failed_temp_files:

                        try:
                            failed_file.unlink()

                        except Exception:
                            pass

                    if attempt < max_retries:
                        wait_time = 5 * attempt

                        print(
                            f"Retrying in {wait_time} seconds..."
                        )

                        time.sleep(wait_time)

            # -----------------------------------------------------
            # Product failed after all retries
            # -----------------------------------------------------

            if not success:

                failed_products += 1

                print(
                    f"PERMANENT FAILURE: {filename}"
                )

        # ---------------------------------------------------------
        # 4. Overall KIC status
        # ---------------------------------------------------------

        if failed_products == 0:
            status = "success"

        elif products_downloaded > 0:
            status = "partial"

        else:
            status = "failed"

        return {
            "kepid": kic,
            "status": status,
            "products_found": products_found,
            "products_downloaded": products_downloaded,
            "failed_products": failed_products
        }

    except Exception as e:

        print(
            f"KIC {kic} search failed: {e}"
        )

        return {
            "kepid": kic,
            "status": "search_failed",
            "products_found": 0,
            "products_downloaded": 0,
            "failed_products": 0
        }

In [17]:
test_kic = 1870398

print("Testing corrected downloader on:", test_kic)

test_result = download_kic_products(test_kic)

print("\nFINAL RESULT:")
print(test_result)

Testing corrected downloader on: 1870398
Already exists: kplr001870398-2009166043257_llc.fits
Already exists: kplr001870398-2009259160929_llc.fits
Already exists: kplr001870398-2009350155506_llc.fits
Already exists: kplr001870398-2010078095331_llc.fits
Already exists: kplr001870398-2010174085026_llc.fits
Already exists: kplr001870398-2010265121752_llc.fits
Already exists: kplr001870398-2010355172524_llc.fits
Already exists: kplr001870398-2011073133259_llc.fits
Already exists: kplr001870398-2011177032512_llc.fits
Already exists: kplr001870398-2011271113734_llc.fits
Already exists: kplr001870398-2012004120508_llc.fits
Already exists: kplr001870398-2012088054726_llc.fits
Already exists: kplr001870398-2012179063303_llc.fits
Already exists: kplr001870398-2012277125453_llc.fits
Already exists: kplr001870398-2013011073258_llc.fits
Already exists: kplr001870398-2013098041711_llc.fits
Already exists: kplr001870398-2013131215648_llc.fits

FINAL RESULT:
{'kepid': 1870398, 'status': 'success', 'pr

In [18]:
fits_files = list(FITS_DIR.glob("*.fits"))

print("FITS files in final directory:", len(fits_files))

for file in fits_files[:20]:
    print(file.name)

FITS files in final directory: 41503
kplr000757450-2009166043257_llc.fits
kplr000757450-2009259160929_llc.fits
kplr000757450-2009350155506_llc.fits
kplr000757450-2010078095331_llc.fits
kplr000757450-2010174085026_llc.fits
kplr000757450-2010265121752_llc.fits
kplr000757450-2010355172524_llc.fits
kplr000757450-2011073133259_llc.fits
kplr000757450-2011177032512_llc.fits
kplr000757450-2011208035123_slc.fits
kplr000757450-2011240104155_slc.fits
kplr000757450-2011271113734_llc.fits
kplr000757450-2011271113734_slc.fits
kplr000757450-2011303113607_slc.fits
kplr000757450-2011334093404_slc.fits
kplr000757450-2012004120508_llc.fits
kplr000757450-2012004120508_slc.fits
kplr000757450-2012032013838_slc.fits
kplr000757450-2012060035710_slc.fits
kplr000757450-2012088054726_llc.fits


In [19]:
pilot_ids = jhalak_ids[:10]

pilot_results = []

for index, kic in enumerate(pilot_ids, start=1):

    print("\n" + "=" * 70)
    print(f"[{index}/10] Processing KIC {kic}")
    print("=" * 70)

    result = download_kic_products(kic)

    pilot_results.append(result)

    print("Result:", result)


[1/10] Processing KIC 757450


Already exists: kplr000757450-2009166043257_llc.fits
Already exists: kplr000757450-2009259160929_llc.fits
Already exists: kplr000757450-2009350155506_llc.fits
Already exists: kplr000757450-2010078095331_llc.fits
Already exists: kplr000757450-2010174085026_llc.fits
Already exists: kplr000757450-2010265121752_llc.fits
Already exists: kplr000757450-2010355172524_llc.fits
Already exists: kplr000757450-2011271113734_slc.fits
Already exists: kplr000757450-2011240104155_slc.fits
Already exists: kplr000757450-2011208035123_slc.fits
Already exists: kplr000757450-2011334093404_slc.fits
Already exists: kplr000757450-2011303113607_slc.fits
Already exists: kplr000757450-2011073133259_llc.fits
Already exists: kplr000757450-2011177032512_llc.fits
Already exists: kplr000757450-2011271113734_llc.fits
Already exists: kplr000757450-2012004120508_slc.fits
Already exists: kplr000757450-2012032013838_slc.fits
Already exists: kplr000757450-2012060035710_slc.fits
Already exists: kplr000757450-2012088054726_sl

In [20]:
pilot_df = pd.DataFrame(pilot_results)

display(pilot_df)

print("Total KICs:", len(pilot_df))
print("Products found:", pilot_df["products_found"].sum())
print("Products downloaded:", pilot_df["products_downloaded"].sum())
print("Products failed:", pilot_df["failed_products"].sum())

,kepid,status,products_found,products_downloaded,failed_products
0,757450,success,35,35,0
1,892772,success,14,14,0
2,1025986,success,19,19,0
3,1026032,success,17,17,0
4,1026957,success,19,19,0
5,1027438,success,18,18,0
6,1161345,success,44,44,0
7,1161949,success,17,17,0
8,1164301,success,15,15,0
9,1293031,success,18,18,0


Total KICs: 10
Products found: 216
Products downloaded: 216
Products failed: 0


In [21]:
import shutil
import time

print("shutil and time imported successfully")

shutil and time imported successfully


In [ ]:
all_results = []

total = len(jhalak_ids)

print(f"Starting full download for {total} KIC IDs...")

for index, kic in enumerate(jhalak_ids, start=1):

    print("\n" + "=" * 70)
    print(f"[{index}/{total}] Processing KIC {kic}")
    print("=" * 70)

    result = download_kic_products(kic)

    all_results.append(result)

    # Save progress after EVERY KIC
    progress_df = pd.DataFrame(all_results)

    progress_df.to_csv(
        METADATA_DIR / "jhalak_download_log.csv",
        index=False
    )

    print("Result:", result)

print("\n" + "=" * 70)
print("FULL DOWNLOAD FINISHED")
print("=" * 70)

final_df = pd.DataFrame(all_results)

print("KICs processed:", len(final_df))
print("\nStatus:")
print(final_df["status"].value_counts())

print("\nProducts found:",
      final_df["products_found"].sum())

print("Products downloaded:",
      final_df["products_downloaded"].sum())

print("Products failed:",
      final_df["failed_products"].sum())

Starting full download for 4107 KIC IDs...

[1/4107] Processing KIC 757450
Already exists: kplr000757450-2009166043257_llc.fits
Already exists: kplr000757450-2009259160929_llc.fits
Already exists: kplr000757450-2009350155506_llc.fits
Already exists: kplr000757450-2010078095331_llc.fits
Already exists: kplr000757450-2010174085026_llc.fits
Already exists: kplr000757450-2010265121752_llc.fits
Already exists: kplr000757450-2010355172524_llc.fits
Already exists: kplr000757450-2011271113734_slc.fits
Already exists: kplr000757450-2011240104155_slc.fits
Already exists: kplr000757450-2011208035123_slc.fits
Already exists: kplr000757450-2011334093404_slc.fits
Already exists: kplr000757450-2011303113607_slc.fits
Already exists: kplr000757450-2011073133259_llc.fits
Already exists: kplr000757450-2011177032512_llc.fits
Already exists: kplr000757450-2011271113734_llc.fits
Already exists: kplr000757450-2012004120508_slc.fits
Already exists: kplr000757450-2012032013838_slc.fits
Already exists: kplr0007